# EGFR network (ODEbase model 1452)

Paper figures, drawn from the saved results. The parameter is the EGF dose, which sets the initial state. The summary figures use the non-negativity correction.

In [ ]:
import os
import sys

sys.path.insert(0, os.path.abspath("../../../"))

import numpy as np

from src import plots as P
from src.run import Experiment, Results
from src.plots import (load_tables, move_corner_label, place_captions, set_font_sizes,
                       set_partition_titles, stack_captions, thin_tick_labels)
from src.hybrid import Hybrid, HybridConfig
# After the imports: src.plots switches matplotlib to Agg.
%matplotlib inline

CASE = "egfr"
OUT_DIR = "."
SWEEP_DIR = "./sweep"
FIG_DIR = "figures"
POSITIVITY = "on"

## Results

In [ ]:
sweep, rank, hybrid = load_tables(OUT_DIR, POSITIVITY)
taus = sorted(np.unique(sweep["test"]["tau_split"]))
etas = sorted(np.unique(sweep["test"]["eta"]), reverse=True)
qoi = [c.split("|", 1)[1] for c in rank["test"].dtype.names if c.startswith("l2-in-time|")]
n_eval = {s: int(np.max(sweep[s]["n_tracers"])) for s in sweep}
# Rank 1 is not shown.
rank = {s: r[r["rank"] > 1] for s, r in rank.items()}

## Errors

In [ ]:
fig_errors, axes = P.sweep_panel_splits(
    sweep["train"], rank["train"], sweep["test"], rank["test"], qoi,
    n_eval_train=n_eval["train"], n_eval_test=n_eval["test"],
    show_tols=tuple(taus), twin_tol=0.005,
    hybrid_train=hybrid["train"], hybrid_test=hybrid["test"],
    ylim=(5e-5, 4e-1), sharex=False, row_gap=0.05, figsize=(20, 11.5), legend_ncol=6, legend_y=0.95,
)
set_font_sizes(fig_errors)
# Failure label positions: (dx, dy, ha, va), offsets in points.
red, blue, orange = "#d62728", "#1f77b4", "#d9601b"
place_captions(axes, {
    (0, 0): {("28%", red, 7e-7): (-6, -10, "right", "top"),
             ("76%", red, 7e-8): (0, 8, "center", "bottom")},
    (1, 0): {("27%", red, 7e-7): (-6, -10, "right", "top"),
             ("75%", red, 7e-8): (0, 8, "center", "bottom")},
    (1, 1): {("0.5%", orange, 1e-4): (8, -12, "left", "top"),
             ("0.5%", orange, 1e-7): (-8, 4, "right", "bottom"),
             ("0.5%", orange, 1e-8): (-6, 3, "right", "bottom"),
             ("0.5%", blue, 1e-8): (-38, 5, "right", "top")},
})

## Clusters, ranks and FOM use

In [ ]:
fig_partition, axes_p = P.sweep_partition_panel(
    sweep["test"], SWEEP_DIR, show_tols=tuple(taus), positivity=POSITIVITY, eval_split="test",
    bar_headroom=1.2,
)
set_font_sizes(fig_partition)
set_partition_titles(fig_partition, axes_p)

## Hybrid error distributions

In [ ]:
fig_violin, _ = P.violin_panel(
    SWEEP_DIR, sorted(taus, reverse=True), qoi, etas=etas, positivity=POSITIVITY, eval_split="test",
    prefix="hybrid_", figsize=(5 * len(taus) + 2, 5 * len(qoi)),
    labels={"EGF-pEGFR-2-pShc-Grb2-SOS-Cbl-CIN85-EPn": "EGFR-EPn"},
)

## Hybrid and non-negativity on two test doses

Left: hybrid against AT-ROM, both without the non-negativity correction. Right: AT-ROM with and without the correction.

In [ ]:
HYBRID_TAU, HYBRID_ETA, HYBRID_DOSE = 1e-3, 1e-5, 158
NONNEG_TAU, NONNEG_ETA, NONNEG_DOSE = 5e-3, 1e-8, 146

exp = Experiment(OUT_DIR, "test")
cfg, ctx = exp.cfg, exp.ctx
_, params, _ = exp.split_files


def load_case(tau, eta, i):
    """The hybrid solver, its arguments for test dose i with the (tau, eta) model, and its FOM times and states."""
    atrom = exp.load_atrom(tau, eta)
    t, y, x = exp.trajectories([i])
    return Hybrid(atrom, cfg.replace(eta=eta), HybridConfig(positivity=False)), (params[i], x[0], ctx.dt_hydro), t[0], y[0]


solver, run, t_h, y_h = load_case(HYBRID_TAU, HYBRID_ETA, HYBRID_DOSE)
hybrid = solver.solve(*run, t_eval=t_h)
rom = solver.solve_pure(*run, t_eval=t_h, positivity=False)
print(hybrid.summary())

solver, run, t_n, y_n = load_case(NONNEG_TAU, NONNEG_ETA, NONNEG_DOSE)
rom_on = solver.solve_pure(*run, t_eval=t_n, positivity=True)
rom_off = solver.solve_pure(*run, t_eval=t_n, positivity=False)

hybrid_title = (rf"Hybrid: $p = {float(np.ravel(params[HYBRID_DOSE])[0]):.3g}\ \mu$M, "
             rf"$\tau_\mathrm{{split}} = 10^{{{np.log10(HYBRID_TAU):.0f}}}$, $\eta = 10^{{{np.log10(HYBRID_ETA):.0f}}}$")
nonneg_title = (rf"Non-negativity: $p = {float(np.ravel(params[NONNEG_DOSE])[0]):.3g}\ \mu$M, "
                rf"$\tau_\mathrm{{split}} = {NONNEG_TAU:g}$, $\eta = 10^{{{np.log10(NONNEG_ETA):.0f}}}$")

common = dict(qoi_indices=ctx.qoi_indices, species=ctx.net.species, xscale="linear",
              labels={"EGF-pEGFR-2-pShc-Grb2-SOS-Cbl-CIN85-EPn": "EGFR–EPn"})
fig_pair, _ = P.qoi_panels_pair(
    dict(result=hybrid, rom=rom, t_ref=t_h, y_ref=y_h, title=hybrid_title, **common),
    dict(rom_on=rom_on, rom_off=rom_off, t_ref=t_n, y_ref=y_n, dt_hydro=ctx.dt_hydro, title=nonneg_title, **common))

## Save

In [ ]:
os.makedirs(FIG_DIR, exist_ok=True)
fig_errors.savefig(f"{FIG_DIR}/panel_errors.pdf", dpi=200, bbox_inches="tight")
fig_partition.savefig(f"{FIG_DIR}/panel_partition.pdf", dpi=200, bbox_inches="tight")
fig_violin.savefig(f"{FIG_DIR}/violin_hybrid.pdf", bbox_inches="tight")
fig_pair.savefig(f"{FIG_DIR}/hybrid_non-neg.pdf", bbox_inches="tight")